In [1]:
from geneformer import EmbExtractor
from geneformer import TranscriptomeTokenizer
import pandas as pd
import numpy as np
import scanpy as sc
import mygene
import os

Everything about Geneformer can be downloaded from huggingface repository:  
https://huggingface.co/ctheodoris/Geneformer

In [5]:
def gene_to_ensembl(gene_names):
    # Initialize the MyGeneInfo client
    mg = mygene.MyGeneInfo()
    
    # Query the database (default species is human)
    results = mg.querymany(gene_names, 
                           scopes='symbol', 
                           fields='ensembl.gene', 
                           species='human',
                           returnall=True)
    
    # Extract Ensembl IDs from results
    mapped = []
    for result in results['out']:
        if 'ensembl' in result:
            # Handle cases where multiple Ensembl IDs exist
            if isinstance(result['ensembl'], list):
                ensembl_ids = [ensembl['gene'] for ensembl in result['ensembl']]
            else:
                ensembl_ids = [result['ensembl']['gene']]
            mapped.append((result['query'], ensembl_ids))
        else:
            mapped.append((result['query'], None))
    
    return mapped

#convert from h5ad to h5ad
def convert_to_ensemble(h5ad_path):
    #load the adata
    try:
        
        adata = sc.read_csv(h5ad_path)
        print("The input path is .csv")
    except:
        print("The input path is .h5ad")
        adata = sc.read_h5ad(h5ad_path)
    #convert the ensemble id
    ensemble_id = gene_to_ensembl(adata.var.index)
    map_id = {i:j[0] for i,j in ensemble_id if j is not None}
    map_id["HIST1H1C"] = "ENSG00000187837"
    map_id["C20orf85"] = "ENSG00000124237"
    #mapping the ensemble id to replace the gene names
    mapped_ids = adata.var.index.map(map_id)
    adata.var["ensembl_id"] = mapped_ids
    adata.obs["n_counts"] = adata.X.sum(axis=1)
    # Getting the sample name
    path_prefix = "/".join(h5ad_path.split("/")[:-2])
    sample_name = os.path.basename(h5ad_path).split(".csv")[0]
    sample_name = os.path.basename(h5ad_path).split(".h5ad")[0]
    
    create_path = os.path.join(path_prefix, "h5ad_data", sample_name)
    os.makedirs(create_path, exist_ok=True)  # Use makedirs for nested folders, and keyword is exist_ok
    
    sample_path = os.path.join(create_path, f"{sample_name}.h5ad")

    adata.write(sample_path)
    return adata


In [14]:
def gen_embeddings(dataname):
    embex = EmbExtractor(model_type="Pretrained",
                     max_ncells=None,
                     forward_batch_size=200,
                     model_version="V2",  # OF NOTE: SET TO V1 MODEL, PROVIDE V1 MODEL PATH IN SUBSEQUENT CODE
                     token_dictionary_file="/home/sxr280/Geneformer/geneformer/token_dictionary_gc104M.pkl",
                     nproc=16)
    # embex = EmbExtractor(model_type="Pretrained",
    #                      max_ncells=None,
    #                      forward_batch_size=4,
    #                      token_dictionary_file="/home/sxr280/Geneformer2/geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl") # change from current default dictionary for 30M model series
    embs = embex.extract_embs("/home/sxr280/Geneformer/Geneformer-V2-316M", 
                              f"/home/sxr280/Geneformer2/tokenization/{dataname}.dataset",
                              "/home/sxr280/Geneformer2/embedding",
                              f"{dataname}")
    
    return embs

Download the data from the link  
https://drive.google.com/drive/folders/12-vuL-gx_wKiLcZlT_ie_mfZEwiNzeND?usp=drive_link
and store it in your own path

In [7]:
adata = convert_to_ensemble("/home/sxr280/Geneformer2/data/VUILD110_ct_train_X.csv")
adata = convert_to_ensemble("/home/sxr280/Geneformer2/data/VUILD110_ct_test_X.csv")
adata = convert_to_ensemble("/home/sxr280/Geneformer2/data/VUILD110_ct_val_X.csv")
adata = convert_to_ensemble("/home/sxr280/Geneformer2/data/VUILD110_nc_train_X.csv")
adata = convert_to_ensemble("/home/sxr280/Geneformer2/data/VUILD110_nc_test_X.csv")
adata = convert_to_ensemble("/home/sxr280/Geneformer2/data/VUILD110_nc_val_X.csv")

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


The input path is .csv


2 input query terms found no hit:	['C20orf85', 'HIST1H1C']
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


The input path is .csv


2 input query terms found no hit:	['C20orf85', 'HIST1H1C']
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


...

2 input query terms found no hit:	['C20orf85', 'HIST1H1C']
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


The input path is .csv


2 input query terms found no hit:	['C20orf85', 'HIST1H1C']
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


The input path is .csv


2 input query terms found no hit:	['C20orf85', 'HIST1H1C']


In [21]:
adata.var

,ensembl_id
ABCC2,ENSG00000023839
ACKR1,ENSG00000213088
ACTA2,ENSG00000107796
AGER,ENSG00000230514
AGR3,ENSG00000173467
...,...
WT1,ENSG00000184937
WWTR1,ENSG00000018408
XBP1,ENSG00000100219
YAP1,ENSG00000137693


Do the tokenization implementation

In [8]:
tk = TranscriptomeTokenizer(nproc=16)
tk.tokenize_data("/home/sxr280/Geneformer2/h5ad_data/VUILD110_ct_train_X", 
                 "/home/sxr280/Geneformer2/tokenization", 
                 "VUILD110_ct_train_X", 
                 file_format="h5ad")

tk.tokenize_data("/home/sxr280/Geneformer2/h5ad_data/VUILD110_ct_test_X", 
                 "/home/sxr280/Geneformer2/tokenization", 
                 "VUILD110_ct_test_X", 
                 file_format="h5ad")

tk.tokenize_data("/home/sxr280/Geneformer2/h5ad_data/VUILD110_ct_val_X", 
                 "/home/sxr280/Geneformer2/tokenization", 
                 "VUILD110_ct_val_X", 
                 file_format="h5ad")

tk.tokenize_data("/home/sxr280/Geneformer2/h5ad_data/VUILD110_nc_train_X", 
                 "/home/sxr280/Geneformer2/tokenization", 
                 "VUILD110_nc_train_X", 
                 file_format="h5ad")

tk.tokenize_data("/home/sxr280/Geneformer2/h5ad_data/VUILD110_nc_test_X", 
                 "/home/sxr280/Geneformer2/tokenization", 
                 "VUILD110_nc_test_X", 
                 file_format="h5ad")

tk.tokenize_data("/home/sxr280/Geneformer2/h5ad_data/VUILD110_nc_val_X", 
                 "/home/sxr280/Geneformer2/tokenization", 
                 "VUILD110_nc_val_X", 
                 file_format="h5ad")

Tokenizing /home/sxr280/Geneformer2/h5ad_data/VUILD110_ct_train_X/VUILD110_ct_train_X.h5ad
/home/sxr280/Geneformer2/h5ad_data/VUILD110_ct_train_X/VUILD110_ct_train_X.h5ad has no column attribute 'filter_pass'; tokenizing all cells.


/home/sxr280/miniconda3/envs/geneformer/lib/python3.10/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/home/sxr280/miniconda3/envs/geneformer/lib/python3.10/site-packages/geneformer/tokenizer.py:547: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


Creating dataset.
Tokenizing /home/sxr280/Geneformer2/h5ad_data/VUILD110_ct_test_X/VUILD110_ct_test_X.h5ad
/home/sxr280/Geneformer2/h5ad_data/VUILD110_ct_test_X/VUILD110_ct_test_X.h5ad has no column attribute 'filter_pass'; tokenizing all cells.


/home/sxr280/miniconda3/envs/geneformer/lib/python3.10/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/home/sxr280/miniconda3/envs/geneformer/lib/python3.10/site-packages/geneformer/tokenizer.py:547: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


Creating dataset.
Tokenizing /home/sxr280/Geneformer2/h5ad_data/VUILD110_ct_val_X/VUILD110_ct_val_X.h5ad
/home/sxr280/Geneformer2/h5ad_data/VUILD110_ct_val_X/VUILD110_ct_val_X.h5ad has no column attribute 'filter_pass'; tokenizing all cells.


...

Creating dataset.
Tokenizing /home/sxr280/Geneformer2/h5ad_data/VUILD110_nc_test_X/VUILD110_nc_test_X.h5ad
/home/sxr280/Geneformer2/h5ad_data/VUILD110_nc_test_X/VUILD110_nc_test_X.h5ad has no column attribute 'filter_pass'; tokenizing all cells.


/home/sxr280/miniconda3/envs/geneformer/lib/python3.10/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/home/sxr280/miniconda3/envs/geneformer/lib/python3.10/site-packages/geneformer/tokenizer.py:547: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


Creating dataset.
Tokenizing /home/sxr280/Geneformer2/h5ad_data/VUILD110_nc_val_X/VUILD110_nc_val_X.h5ad
/home/sxr280/Geneformer2/h5ad_data/VUILD110_nc_val_X/VUILD110_nc_val_X.h5ad has no column attribute 'filter_pass'; tokenizing all cells.


/home/sxr280/miniconda3/envs/geneformer/lib/python3.10/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/home/sxr280/miniconda3/envs/geneformer/lib/python3.10/site-packages/geneformer/tokenizer.py:547: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


Creating dataset.


generating the embeddings

In [15]:
embs = gen_embeddings("VUILD110_ct_train_X")
embs = gen_embeddings("VUILD110_ct_test_X")
embs = gen_embeddings("VUILD110_ct_val_X")
embs = gen_embeddings("VUILD110_nc_train_X")
embs = gen_embeddings("VUILD110_nc_test_X")
embs = gen_embeddings("VUILD110_nc_val_X")

  0%|          | 0/401 [00:00<?, ?it/s]

  0%|          | 0/51 [00:00<?, ?it/s]

  0%|          | 0/51 [00:00<?, ?it/s]

  0%|          | 0/401 [00:00<?, ?it/s]

  0%|          | 0/51 [00:00<?, ?it/s]

  0%|          | 0/51 [00:00<?, ?it/s]

Converting the ".csv" embedding file to ".npy"

In [16]:
import numpy as np
import pandas as pd


In [18]:
VUILD110_ct_train_X = pd.read_csv("/home/sxr280/Geneformer2/embedding/VUILD110_ct_train_X.csv", index_col=0)
VUILD110_ct_test_X = pd.read_csv("/home/sxr280/Geneformer2/embedding/VUILD110_ct_test_X.csv", index_col=0)
VUILD110_ct_val_X = pd.read_csv("/home/sxr280/Geneformer2/embedding/VUILD110_ct_val_X.csv", index_col=0)
VUILD110_nc_train_X = pd.read_csv("/home/sxr280/Geneformer2/embedding/VUILD110_nc_train_X.csv", index_col=0)
VUILD110_nc_test_X = pd.read_csv("/home/sxr280/Geneformer2/embedding/VUILD110_nc_test_X.csv", index_col=0)
VUILD110_nc_val_X = pd.read_csv("/home/sxr280/Geneformer2/embedding/VUILD110_nc_val_X.csv", index_col=0)



In [19]:
np.save("/home/sxr280/Geneformer2/embedding/Geneformer_VUILD110_ct_train_embed_316M.npy", VUILD110_ct_train_X)
np.save("/home/sxr280/Geneformer2/embedding/Geneformer_VUILD110_ct_test_embed_316M.npy", VUILD110_ct_test_X)
np.save("/home/sxr280/Geneformer2/embedding/Geneformer_VUILD110_ct_val_embed_316M.npy", VUILD110_ct_val_X)
np.save("/home/sxr280/Geneformer2/embedding/Geneformer_VUILD110_nc_train_embed_316M.npy", VUILD110_nc_train_X)
np.save("/home/sxr280/Geneformer2/embedding/Geneformer_VUILD110_nc_test_embed_316M.npy", VUILD110_nc_test_X)
np.save("/home/sxr280/Geneformer2/embedding/Geneformer_VUILD110_nc_val_embed_316M.npy", VUILD110_nc_val_X)